# VN-HSGExam — Parse 48 file đề/đáp án thành JSON

## Cell 1 — Tạo 2 file Python parser

In [ ]:
# Tạo file vn_hsgexam_parser.py
parser_code = '''
from __future__ import annotations
import json, re
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Optional

SUBJECT_RANGES = {
    "history":   (1,   40),
    "geography": (41,  80),
    "civics":    (81, 120),
    "biology":   (81, 120),
}
DIFFICULTY_BUCKETS = {
    "history":   (20, 30),
    "geography": (60, 70),
    "civics":    (100, 110),
    "biology":   (100, 110),
}
VALID_TAGS = {"ATLAS", "HÌNH", "PHẢ HỆ", "BẢNG", "ĐỒ THỊ"}

RE_QUESTION_START = re.compile(r"^Câu\\s+(\\d+)\\s*:\\s*(.*)$")
RE_CHOICE = re.compile(r"^([ABCD])\\.\\s*(.*)$")
RE_EXCLUSION = re.compile(r"^\\[LOẠI:\\s*([^\\]]+)\\]\\s*$")
RE_FOOTER = re.compile(r"^=+\\s*$")
RE_ANSWER = re.compile(r"^\\s*(\\d+)\\s*[.:]\\s*([ABCD])\\s*$")

@dataclass
class Question:
    question_id: str
    subject: str
    year: int
    question_number: int
    stem: str
    choices: dict
    answer: Optional[str]
    is_usable: bool
    exclusion_tag: Optional[str]
    difficulty: str
    metadata: dict = field(default_factory=dict)

def difficulty_for(subject, qnum):
    low_max, mid_max = DIFFICULTY_BUCKETS[subject]
    if qnum <= low_max: return "easy"
    if qnum <= mid_max: return "medium"
    return "hard"

def normalize_tag(raw_tag):
    return re.sub(r"\\s+", " ", raw_tag).strip().upper()

def parse_answer_key(path):
    answers = {}
    with open(path, "r", encoding="utf-8") as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.strip()
            if not line: continue
            m = RE_ANSWER.match(line)
            if not m:
                raise ValueError(f"{path.name}:{lineno}: cannot parse answer line: {raw!r}")
            qnum = int(m.group(1))
            letter = m.group(2)
            if qnum in answers:
                raise ValueError(f"{path.name}:{lineno}: duplicate answer for question {qnum}")
            answers[qnum] = letter
    return answers

class _QuestionBuffer:
    def __init__(self, qnum, first_stem_line):
        self.qnum = qnum
        self.stem_lines = [first_stem_line] if first_stem_line else []
        self.choices = {}
        self._current_choice = None
        self.exclusion_tag = None

    def add_line(self, line):
        m_tag = RE_EXCLUSION.match(line)
        if m_tag:
            self.exclusion_tag = normalize_tag(m_tag.group(1))
            self._current_choice = None
            return
        m_choice = RE_CHOICE.match(line)
        if m_choice:
            letter = m_choice.group(1)
            content = m_choice.group(2)
            self.choices[letter] = [content] if content else []
            self._current_choice = letter
            return
        if self._current_choice is not None:
            self.choices[self._current_choice].append(line)
        else:
            self.stem_lines.append(line)

    def finalize(self, subject, year, answer):
        missing = [c for c in "ABCD" if c not in self.choices]
        if missing:
            raise ValueError(f"Question {self.qnum}: missing choices {missing}")
        stem = "\\n".join(s for s in self.stem_lines).strip()
        choices = {letter: " ".join(parts).strip() for letter, parts in self.choices.items()}
        if self.exclusion_tag is not None and self.exclusion_tag not in VALID_TAGS:
            raise ValueError(f"Question {self.qnum}: unknown exclusion tag {self.exclusion_tag!r}")
        is_usable = self.exclusion_tag is None
        return Question(
            question_id=f"{subject}_{year}_q{self.qnum}",
            subject=subject, year=year, question_number=self.qnum,
            stem=stem, choices=choices, answer=answer,
            is_usable=is_usable, exclusion_tag=self.exclusion_tag,
            difficulty=difficulty_for(subject, self.qnum),
            metadata={"stem_char_length": len(stem), "stem_line_count": len(self.stem_lines)},
        )

def parse_question_file(path, subject, year, answers):
    questions = []
    current = None
    in_footer = False
    def flush():
        if current is not None:
            ans = answers.get(current.qnum)
            questions.append(current.finalize(subject, year, ans))
    with open(path, "r", encoding="utf-8") as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.rstrip("\\n").rstrip("\\r")
            stripped = line.strip()
            if RE_FOOTER.match(stripped):
                flush()
                current = None
                in_footer = True
                continue
            if in_footer: continue
            if not stripped:
                if current is not None:
                    current._current_choice = None
                continue
            m_q = RE_QUESTION_START.match(stripped)
            if m_q:
                flush()
                qnum = int(m_q.group(1))
                first_stem = m_q.group(2).strip()
                current = _QuestionBuffer(qnum, first_stem)
                continue
            if current is None: continue
            try:
                current.add_line(stripped)
            except Exception as e:
                raise ValueError(f"{path.name}:{lineno}: {e}") from e
    flush()
    return questions

def validate_question_set(questions, subject, year, answers):
    issues = []
    low, high = SUBJECT_RANGES[subject]
    expected = set(range(low, high + 1))
    got = {q.question_number for q in questions}
    missing = expected - got
    extra = got - expected
    if missing: issues.append(f"Missing question numbers: {sorted(missing)}")
    if extra: issues.append(f"Unexpected question numbers: {sorted(extra)}")
    ans_missing = expected - set(answers.keys())
    ans_extra = set(answers.keys()) - expected
    if ans_missing: issues.append(f"Answer key missing for: {sorted(ans_missing)}")
    if ans_extra: issues.append(f"Answer key has extra entries: {sorted(ans_extra)}")
    for q in questions:
        for letter in "ABCD":
            if letter not in q.choices or not q.choices[letter]:
                issues.append(f"Q{q.question_number}: empty/missing choice {letter}")
        if q.is_usable and q.answer is None:
            issues.append(f"Q{q.question_number}: usable but no answer in key")
    return issues

def parse_pair(question_path, answer_path, subject, year):
    answers = parse_answer_key(answer_path)
    questions = parse_question_file(question_path, subject, year, answers)
    issues = validate_question_set(questions, subject, year, answers)
    return questions, issues

def questions_to_json(questions):
    return [asdict(q) for q in questions]
'''

with open("vn_hsgexam_parser.py", "w", encoding="utf-8") as f:
    f.write(parser_code)

# Tạo file batch_parse.py
batch_code = '''
from __future__ import annotations
import argparse, json, sys
from dataclasses import asdict
from pathlib import Path
from collections import Counter, defaultdict
from vn_hsgexam_parser import parse_pair, SUBJECT_RANGES

SUBJECTS = ["history", "geography", "civics", "biology"]
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]

def find_pair(input_dir, subject, year):
    qp = input_dir / f"ques_{subject}_{year}.txt"
    ap = input_dir / f"ans_{subject}_{year}.txt"
    if qp.exists() and ap.exists():
        return qp, ap
    return None

def compute_stats(questions):
    by_subject = defaultdict(list)
    for q in questions:
        by_subject[q.subject].append(q)
    out = {"by_subject": {}, "by_year": {}, "by_subject_year": {}}
    for subj, qs in by_subject.items():
        usable = [q for q in qs if q.is_usable]
        diffs = Counter(q.difficulty for q in usable)
        ans = Counter(q.answer for q in usable if q.answer)
        out["by_subject"][subj] = {
            "total": len(qs), "usable": len(usable),
            "excluded": len(qs) - len(usable),
            "difficulty": dict(diffs), "answer_dist": dict(ans),
            "excluded_tags": dict(Counter(q.exclusion_tag for q in qs if q.exclusion_tag)),
        }
    by_year = defaultdict(list)
    for q in questions:
        by_year[q.year].append(q)
    for yr, qs in by_year.items():
        usable = [q for q in qs if q.is_usable]
        out["by_year"][str(yr)] = {"total": len(qs), "usable": len(usable)}
    for subj in SUBJECTS:
        for yr in YEARS:
            qs = [q for q in questions if q.subject == subj and q.year == yr]
            if qs:
                out["by_subject_year"][f"{subj}_{yr}"] = {
                    "total": len(qs), "usable": sum(1 for q in qs if q.is_usable),
                }
    return out

def render_markdown_report(report, stats):
    lines = ["# VN-HSGExam Parse Report\\n"]
    lines.append(f"- Input directory: `{report['input_dir']}`")
    lines.append(f"- Missing files: {len(report['missing'])}")
    if report["missing"]:
        for m in report["missing"]:
            lines.append(f"    - {m}")
    lines.append("")
    lines.append("## Per-Subject Summary\\n")
    lines.append("| Subject | Total | Usable | Excluded | Tags |")
    lines.append("|---|---:|---:|---:|---|")
    for subj in SUBJECTS:
        s = stats["by_subject"].get(subj)
        if not s: continue
        tags = ", ".join(f"{k}:{v}" for k, v in s.get("excluded_tags", {}).items()) or "—"
        lines.append(f"| {subj} | {s['total']} | {s['usable']} | {s['excluded']} | {tags} |")
    lines.append("\\n## Per-File Status\\n")
    lines.append("| File | Parsed | Usable | Issues |")
    lines.append("|---|---:|---:|---|")
    for key, info in sorted(report["files"].items()):
        if "error" in info:
            lines.append(f"| {key} | ERROR | — | {info['error']} |")
        else:
            iss = "; ".join(info["issues"]) if info["issues"] else "✓"
            lines.append(f"| {key} | {info['parsed_count']} | {info['usable_count']} | {iss} |")
    lines.append("\\n## Difficulty Distribution (usable only)\\n")
    lines.append("| Subject | Easy | Medium | Hard |")
    lines.append("|---|---:|---:|---:|")
    for subj in SUBJECTS:
        s = stats["by_subject"].get(subj)
        if not s: continue
        d = s["difficulty"]
        lines.append(f"| {subj} | {d.get('easy',0)} | {d.get('medium',0)} | {d.get('hard',0)} |")
    return "\\n".join(lines) + "\\n"

def run_batch(input_dir, output_dir):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    all_questions = []
    report = {"input_dir": str(input_dir), "files": {}, "missing": []}
    total_issues = 0
    for subject in SUBJECTS:
        for year in YEARS:
            key = f"{subject}_{year}"
            pair = find_pair(input_dir, subject, year)
            if pair is None:
                report["missing"].append(key)
                print(f"  [MISSING] {key}")
                continue
            qp, ans_p = pair
            try:
                questions, issues = parse_pair(qp, ans_p, subject, year)
            except Exception as e:
                report["files"][key] = {"error": str(e)}
                total_issues += 1
                print(f"  [ERROR] {key}: {e}")
                continue
            usable = sum(1 for q in questions if q.is_usable)
            tag_counts = Counter(q.exclusion_tag for q in questions if q.exclusion_tag)
            report["files"][key] = {
                "parsed_count": len(questions), "usable_count": usable,
                "excluded_count": len(questions) - usable,
                "excluded_tags": dict(tag_counts), "issues": issues,
            }
            total_issues += len(issues)
            all_questions.extend(questions)
            status = "OK" if not issues else f"WARN ({len(issues)} issues)"
            print(f"  [{status}] {key}: {len(questions)} parsed, {usable} usable, {len(questions)-usable} excluded")
            for iss in issues:
                print(f"      - {iss}")
    stats = compute_stats(all_questions)
    report["overall"] = stats
    all_json = [asdict(q) for q in all_questions]
    usable_json = [asdict(q) for q in all_questions if q.is_usable]
    (output_dir / "vn_hsgexam_all.json").write_text(json.dumps(all_json, ensure_ascii=False, indent=2), encoding="utf-8")
    (output_dir / "vn_hsgexam_usable.json").write_text(json.dumps(usable_json, ensure_ascii=False, indent=2), encoding="utf-8")
    (output_dir / "parse_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    (output_dir / "parse_report.md").write_text(render_markdown_report(report, stats), encoding="utf-8")
    print()
    print("=" * 60)
    print(f"  Total parsed     : {len(all_questions)}")
    print(f"  Total usable     : {sum(1 for q in all_questions if q.is_usable)}")
    print(f"  Total excluded   : {sum(1 for q in all_questions if not q.is_usable)}")
    print(f"  Missing files    : {len(report['missing'])}")
    print(f"  Files with issues: {total_issues}")
    print(f"  Output dir       : {output_dir}")
    print("=" * 60)
    return all_questions, report
'''

with open("batch_parse.py", "w", encoding="utf-8") as f:
    f.write(batch_code)

print("✓ Đã tạo 2 file: vn_hsgexam_parser.py + batch_parse.py")
print("  Sẵn sàng cho cell 2 (upload file).")

## Cell 2 — Upload 48 file .txt

In [ ]:
# ============================================================
# CELL 2 — Mount Google Drive và copy 48 file vào ./raw/
# ============================================================
import os
import shutil
from google.colab import drive

# Bước 2.1 — Mount Drive
# Khi chạy lần đầu, Colab sẽ mở popup xin quyền truy cập Drive
# → Click vào link, đăng nhập Google, copy authorization code, paste vào ô
drive.mount('/content/drive')

# Bước 2.2 — Trỏ đến folder Drive chứa 48 file raw
# ⚠️ SỬA dòng dưới: đổi 'VN-HSGExam-Raw' thành đúng tên folder bạn tạo trên Drive
DRIVE_FOLDER = '/content/drive/MyDrive/VN-HSGExam/01_raw_txts'

# Bước 2.3 — Kiểm tra folder có tồn tại không
if not os.path.exists(DRIVE_FOLDER):
    print(f"❌ Không tìm thấy folder: {DRIVE_FOLDER}")
    print(f"\n   Các folder có sẵn trong MyDrive:")
    for f in sorted(os.listdir('/content/drive/MyDrive')):
        if os.path.isdir(f'/content/drive/MyDrive/{f}'):
            print(f"     - {f}")
    print(f"\n   Sửa biến DRIVE_FOLDER ở trên cho đúng tên rồi chạy lại cell này.")
else:
    # Bước 2.4 — Copy tất cả file .txt từ Drive vào ./raw/ trên Colab
    os.makedirs('raw', exist_ok=True)
    drive_files = sorted(f for f in os.listdir(DRIVE_FOLDER) if f.endswith('.txt'))

    for fname in drive_files:
        src = os.path.join(DRIVE_FOLDER, fname)
        dst = os.path.join('raw', fname)
        shutil.copy(src, dst)

    # Bước 2.5 — Verify
    txt_files = sorted(f for f in os.listdir('raw') if f.endswith('.txt'))
    print(f"\n✓ Đã copy {len(txt_files)} file từ Drive vào ./raw/")
    print(f"  Mong đợi: 48 file (4 môn × 6 năm × 2 file)")

    if len(txt_files) != 48:
        print(f"\n  ⚠ Số file không đúng. Danh sách:")
        for f in txt_files:
            print(f"    {f}")
    else:
        # Group theo môn cho dễ xem
        from collections import defaultdict
        by_subj = defaultdict(list)
        for f in txt_files:
            parts = f.replace('.txt', '').split('_')
            if len(parts) >= 3:
                subject = parts[1]
                by_subj[subject].append(f)

        for subj in ['history', 'geography', 'civics', 'biology']:
            files = sorted(by_subj.get(subj, []))
            print(f"  {subj}: {len(files)} file")

## Cell 3 — Chạy parser

In [ ]:
from batch_parse import run_batch

all_questions, report = run_batch("raw", "parsed")

print("\n--- File outputs ---")
for f in sorted(os.listdir("parsed")):
    size = os.path.getsize(f"parsed/{f}")
    print(f"  parsed/{f}  ({size:,} bytes)")

## Cell 4 — Xem báo cáo trước khi download

In [ ]:
with open("parsed/parse_report.md", "r", encoding="utf-8") as f:
    print(f.read())

## Cell 5 — Download kết quả về máy

In [ ]:
from google.colab import files

for f in ["vn_hsgexam_all.json", "vn_hsgexam_usable.json",
         "parse_report.json", "parse_report.md"]:
    files.download(f"parsed/{f}")